In [2]:
import torch
import os
import numpy as np
import torch.nn as nn
from torch.utils.data import TensorDataset, DataLoader
import copy
from pathlib import Path
import torch.profiler
from enum import Enum
import optuna
from sklearn.model_selection import GroupShuffleSplit
import random
from sklearn.metrics import (
    accuracy_score,
    f1_score,
)
import polars as pl
import time

torch.set_float32_matmul_precision('high')
torch.backends.cuda.matmul.allow_tf32 = False
torch.backends.cudnn.allow_tf32 = False
torch.backends.cudnn.deterministic = True
torch.backends.cudnn.benchmark = False
torch.use_deterministic_algorithms(True)

In [3]:
def seed_all(seed=42):
    os.environ["PYTHONHASHSEED"] = str(seed)

    random.seed(seed)
    np.random.seed(seed)

    torch.manual_seed(seed)
    torch.cuda.manual_seed_all(seed)

seed_all()

In [4]:
class ModelType(Enum):
    CNN = 1
    LSTM = 2
    CNN_LSTM = 3
    LSTM_CNN = 4
    CNN_LSTM_Fusion = 5
    CNN_Transformer = 6
    MLP = 7

class EvalMetric(Enum):
    accuracy = 1
    F1 = 2
    custom = 3

class ExperimentType(Enum):
    fixed_model_params = 1
    best_model = 2

In [5]:
device = torch.device(
    "cuda" if torch.cuda.is_available() else "cpu"
)

torch.backends.cudnn.benchmark = True

print(device)

cuda


In [6]:
root = "../../../"
data_path = f'{root}data/'
training_data_path = data_path + "Final Training Data/"

In [7]:
class CNN(nn.Module):
    def __init__(self, input_channels, output_channels, num_classes, activation_fn, dropout, output_c_multip):
        super().__init__()

        C1 = output_channels
        C2 = C1*output_c_multip
        C3 = C2*output_c_multip

        self.features = nn.Sequential(
            nn.Conv1d(input_channels, C1, 5, padding=2),
            nn.BatchNorm1d(C1),
            activation_fn(),

            nn.Conv1d(C1, C2, 5, padding=2),
            nn.BatchNorm1d(C2),
            activation_fn(),

            nn.Conv1d(C2, C3, 3, padding=1),
            nn.BatchNorm1d(C3),
            activation_fn(),
        )

        self.pool = nn.AdaptiveAvgPool1d(1)

        self.classifier = nn.Sequential(
            nn.Linear(C3, C3),
            activation_fn(),
            nn.Dropout(dropout),
            nn.Linear(C3, num_classes)
        )

    def forward(self, x):
        x = x.permute(0, 2, 1)
        x = self.features(x)
        x = self.pool(x).squeeze(-1)
        return self.classifier(x)



        
class LSTM(nn.Module):
    def __init__(self, input_size, hidden_size, num_layers, num_classes, dropout):
        super().__init__()

        H = hidden_size  

        self.lstm = nn.LSTM(
            input_size=input_size,
            hidden_size=H,
            num_layers=num_layers,   
            batch_first=True,
            dropout=dropout
        )

        self.head = nn.Sequential(
            nn.Linear(H, H),
            nn.ReLU(),
            nn.Dropout(dropout),
            nn.Linear(H, num_classes)
        )

    def forward(self, x):
        x, _ = self.lstm(x)
        x = x.mean(dim=1)
        return self.head(x)

class AttentionPooling(nn.Module):
    def __init__(self, hidden_dim):
        super().__init__()
        self.attn = nn.Linear(hidden_dim, 1)

    def forward(self, x):

        weights = self.attn(x)             
        weights = torch.softmax(weights, dim=1)

        pooled = torch.sum(x * weights, dim=1)
        return pooled





class CNN_LSTM(nn.Module):
    def __init__(self, input_channels, output_channels, hidden_size, num_layers, num_classes, activation_fn, dropout, output_c_multip):
        super().__init__()

        C1 = output_channels
        C2 = C1 * output_c_multip
        C3 = C2 * output_c_multip
        H = hidden_size  

        self.cnn = nn.Sequential(
            nn.Conv1d(input_channels, C1, kernel_size=5, padding=2),
            nn.BatchNorm1d(C1),
            activation_fn(),
            nn.Dropout(dropout),

            nn.Conv1d(C1, C2, kernel_size=5, padding=2),
            nn.BatchNorm1d(C2),
            activation_fn(),
            nn.Dropout(dropout),

            nn.Conv1d(C2, C3, kernel_size=3, padding=1),
            nn.BatchNorm1d(C3),
            activation_fn(),

            nn.MaxPool1d(2)
        )

        self.lstm = nn.LSTM(
            input_size=C3,
            hidden_size=H,
            num_layers=num_layers,
            batch_first=True,
            dropout=dropout
        )

        self.norm = nn.LayerNorm(H)

        self.pool = AttentionPooling(H)

        self.head = nn.Sequential(
            nn.Linear(H, H),
            nn.ReLU(),
            nn.Dropout(dropout),

            nn.Linear(H, H // 2),
            nn.ReLU(),
            nn.Dropout(dropout),

            nn.Linear(H // 2, num_classes)
        )

    def forward(self, x):
        # x: (B, T, F)
        x = x.permute(0, 2, 1)  
        x = self.cnn(x)         
        x = x.permute(0, 2, 1)  
        
        x, _ = self.lstm(x)      
        x = self.norm(x)
        x = self.pool(x)         

        return self.head(x)




class CNN_Transformer(nn.Module):
    def __init__(self, input_channels, output_channels, num_layers, num_classes, feature_dim, activation_fn, dropout, output_c_multip):
        super().__init__()

        C1 = output_channels
        C2 = C1*output_c_multip
        C3 = C2*output_c_multip

        self.cnn = nn.Sequential(
            nn.Conv1d(input_channels, C1, 5, padding=2),
            nn.BatchNorm1d(C1),
            activation_fn(),

            nn.Conv1d(C1, C2, 5, padding=2),
            nn.BatchNorm1d(C2),
            activation_fn(),

            nn.Conv1d(C2, C3, 5, padding=2),
            nn.BatchNorm1d(C3),
            activation_fn(),

            nn.MaxPool1d(2)
        )

        self.transformer = nn.TransformerEncoder(
            nn.TransformerEncoderLayer(
                d_model=C3,
                nhead=4,
                batch_first=True
            ),
            num_layers=num_layers
        )

        self.fusion = nn.Sequential(
            nn.Linear(C3 + feature_dim, 256),
            nn.GELU(),
            nn.Dropout(dropout),
            nn.Linear(256, num_classes)
        )

    def forward(self, x, x_engineered):
        x = x.permute(0, 2, 1)
        x = self.cnn(x)
        x = x.permute(0, 2, 1)

        x = self.transformer(x)
        x = x.mean(dim=1)

        x = torch.cat([x, x_engineered], dim=1)
        return self.fusion(x)



        
class LSTM_CNN(nn.Module):
    def __init__(self, input_size, hidden_size, num_layers, num_classes, activation_fn, dropout):
        super().__init__()

        H = hidden_size

        self.lstm = nn.LSTM(
            input_size=input_size,
            hidden_size=H,
            num_layers=num_layers,
            batch_first=True
        )

        self.cnn = nn.Sequential(
            nn.Conv1d(H, H, 5, padding=2),
            nn.BatchNorm1d(H),
            activation_fn(),

            nn.MaxPool1d(2),

            nn.Conv1d(H, H, 3, padding=1),
            activation_fn(),
        )

        self.pool = nn.AdaptiveAvgPool1d(1)

        self.head = nn.Sequential(
            nn.Linear(H, H),
            activation_fn(),
            nn.Dropout(dropout),
            nn.Linear(H, num_classes)
        )

    def forward(self, x):
        x, _ = self.lstm(x)
        x = x.permute(0, 2, 1)
        x = self.cnn(x)
        x = self.pool(x).squeeze(-1)
        return self.head(x)





class CNN_LSTM_Fusion(nn.Module):
    def __init__(self, input_channels, output_channels, hidden_size, num_layers, num_classes, activation_fn, dropout, output_c_multip):
        super().__init__()

        C1 = output_channels
        C2 = C1 * output_c_multip
        H = hidden_size  

        # CNN branch
        self.cnn = nn.Sequential(
            nn.Conv1d(input_channels, C1, 5, padding=2),
            nn.BatchNorm1d(C1),
            activation_fn(),

            nn.Conv1d(C1, C2, kernel_size=5, padding=2),
            nn.BatchNorm1d(C2),
            activation_fn(),

            nn.MaxPool1d(2),
        )

        # LSTM branch
        self.lstm = nn.LSTM(
            input_size=input_channels,
            hidden_size=H,
            num_layers=num_layers,
            batch_first=True
        )

        # fusion head
        self.classifier = nn.Sequential(
            nn.Linear(C2 + H, 96),
            activation_fn(),
            nn.Dropout(dropout),
            nn.Linear(96, num_classes)
        )

    def forward(self, x):
        # CNN
        x_cnn = x.permute(0, 2, 1)
        x_cnn = self.cnn(x_cnn)
        x_cnn = x_cnn.mean(dim=-1)

        # LSTM
        x_lstm, _ = self.lstm(x)
        x_lstm = x_lstm.mean(dim=1)

        x = torch.cat([x_cnn, x_lstm], dim=1)
        return self.classifier(x)



        
class MLP(nn.Module):
    def __init__(self, input_features, output_channles, num_classes, activation_fn, dropout, output_c_multip):
        super().__init__()

        C1 = output_channles
        C2 = C1*output_c_multip

        self.net = nn.Sequential(
            nn.Linear(input_features, C1),
            nn.LayerNorm(C1),
            activation_fn(),
            nn.Dropout(dropout),

            nn.Linear(C1, C2),
            nn.LayerNorm(C2),
            activation_fn(),
            nn.Dropout(dropout),

            nn.Linear(C2, num_classes)
        )

    def forward(self, x):
        return self.net(x)

In [8]:
def count_parameters(model):
    return sum(p.numel() for p in model.parameters() if p.requires_grad)

In [9]:
def compute_accuracy(y_true, y_pred):
    return accuracy_score(y_true, y_pred)

def compute_f1_score(y_true, y_pred):
    return f1_score(
            y_true, y_pred,
            average="macro",
            zero_division=0
    )

In [10]:
def load_npz(path):
    path = Path(path)
    data = np.load(f"{path}.npz", allow_pickle=True)
    return data["X"], data["y"]

def load_subject(cache_dir, subject, use_raw, use_engineered):
    X_raw = None
    X_engineered = None
    y = None

    cache_dir = Path(cache_dir)
    if use_raw:
        X_raw, y = load_npz(cache_dir / subject / f"{subject}_raw")

    if use_engineered:
        X_engineered, y_engineered = load_npz(cache_dir / subject / f"{subject}_extracted")

        if y is None:
            y = y_engineered
        elif not np.array_equal(y, y_engineered):
            raise ValueError("Labels do not match")

    return X_raw, X_engineered, y

def get_subjects(cache_dir, use_raw, use_engineered):
    cache_dir = Path(cache_dir)
    subjects = set()

    if use_raw:
        subjects.update(
            p.stem.removesuffix("_raw")
            for p in Path(cache_dir).rglob("*_raw.npz")
        )

    if use_engineered:
        subjects.update(
            p.stem.removesuffix("_extracted")
            for p in Path(cache_dir).rglob("*_extracted.npz")
        )

    return sorted(subjects)

In [11]:
def get_loader(settings, X_train_raw, X_train_extracted, y_train):
    sampler = None
    shuffle = True
    batch_size = settings["batch size"]
    if settings["weight"] == "weighted":
            
        class_counts = np.bincount(y_train)
        weights = class_counts.sum() / class_counts
        
        weights = weights / weights.mean()
        #weights = 1.0 / class_counts
        
        weights = torch.tensor(weights, dtype=torch.float32).to(device)

        loss_fn = torch.nn.CrossEntropyLoss(weight=weights)
    else:
        loss_fn = torch.nn.CrossEntropyLoss(label_smoothing = settings["label smoothing"])

    if X_train_raw is None:
        X_train_raw = torch.empty((len(y_train), 0, 0), dtype=torch.float32)

    if X_train_extracted is None:
        X_train_extracted = torch.empty((len(y_train), 0), dtype=torch.float32)

    return DataLoader(
                    TensorDataset(X_train_raw, X_train_extracted, y_train),
                    batch_size=batch_size,
                    shuffle=shuffle,
                    sampler=sampler,
                    #num_workers=1,
                    pin_memory=True,
                    #prefetch_factor=2
                ), loss_fn

In [12]:
# Gradient Accumulation
def train_epoch(model, loader, optimizer, loss_fn, accumulation_steps=64):
    model.train()
    optimizer.zero_grad()
    for i, (X_batch_raw, X_batch_extracted, y_batch) in enumerate(loader):
        X_batch_raw = X_batch_raw.to(device, non_blocking=True)
        X_batch_extracted = X_batch_extracted.to(device, non_blocking=True)
        y_batch = y_batch.to(device, non_blocking=True)

        torch.compiler.cudagraph_mark_step_begin()
        
        with torch.amp.autocast(device_type="cuda"):
            logits = model_forward(model, X_batch_raw, X_batch_extracted)
            loss = loss_fn(logits, y_batch)
            loss = loss / accumulation_steps
        
        loss.backward()
        
        if (i + 1) % accumulation_steps == 0:
            optimizer.step()
            optimizer.zero_grad()

def validate_epoch(model, X_val_raw, X_val_extracted, y_val, metadata):

    model.eval()

    with torch.no_grad():

        X_val_raw = X_val_raw.to(device)
        X_val_extracted = X_val_extracted.to(device)
        y_val = y_val.to(device)

        logits = model_forward(
            model,
            X_val_raw,
            X_val_extracted
        )

        probs = torch.softmax(logits, dim=1)

        preds = torch.argmax(probs, dim=1)
        preds = preds.cpu().numpy()

        if metadata["eval_metric"] == EvalMetric.accuracy:
            val_metric = compute_accuracy(
                y_val.cpu(),
                preds
            )

        elif metadata["eval_metric"] == EvalMetric.F1:
            val_metric = compute_f1_score(
                y_val.cpu(),
                preds
            )
        else:
            confidences, preds = torch.max(probs, dim=1)

            mask = confidences >= 0.8

            if mask.sum() == 0:
                val_metric = 0
            else:
                coverage = mask.float().mean().item()
                precision = (
                    preds[mask] == y_val[mask]
                ).float().mean().item()

                val_metric = precision * coverage

    return val_metric

def model_forward(model, x_raw, x_extracted):

    has_raw = x_raw.numel() > 0
    has_extracted = x_extracted.numel() > 0

    if has_raw and has_extracted:
        return model(x_raw, x_extracted)

    if has_raw:
        return model(x_raw)

    if has_extracted:
        return model(x_extracted)

    raise ValueError("No input features provided")

In [13]:
def get_model(model_settings, X_train_main_raw, X_train_main_extracted, y_train_main):
    match model_settings["activation_fn"]:
        case "relu":
            activation_fn = nn.ReLU
        case "gelu":
            activation_fn = nn.GELU

    dropout = model_settings["dropout"]
    hidden_size = model_settings["hidden_size"]
    num_layers = model_settings["num_layers"]
    output_channles = model_settings["output_channels"]
    output_c_multip = model_settings["output_c_multip"]
    
    input_channels_raw = X_train_main_raw.shape[-1]
    input_channels_extracted = X_train_main_extracted.shape[1]
    num_classes = len(torch.unique(y_train_main))
    
    match model_settings["model"]:
        case ModelType.CNN:
            model = CNN(
                input_channels_raw,
                output_channles,
                num_classes,
                activation_fn,
                dropout,
                output_c_multip
            ).to(device)
        case ModelType.LSTM:
            model = LSTM(
                input_channels_raw,
                hidden_size,
                num_layers,
                num_classes,
                dropout
            ).to(device)
        case ModelType.CNN_LSTM:
            model = CNN_LSTM(
                input_channels_raw,
                output_channles,
                hidden_size, 
                num_layers,
                num_classes,
                activation_fn,
                dropout,
                output_c_multip
            ).to(device)
        case ModelType.LSTM_CNN:
            model = LSTM_CNN(
                input_channels_raw,
                hidden_size, 
                num_layers, 
                num_classes, 
                activation_fn, 
                dropout
            ).to(device)
        case ModelType.CNN_Transformer:
            model = CNN_Transformer(
                input_channels_raw,
                output_channles,
                num_layers,
                num_classes,
                input_channels_extracted,
                activation_fn,
                dropout,
                output_c_multip
            ).to(device)
        case ModelType.CNN_LSTM_Fusion:
            model = CNN_LSTM_Fusion(
                input_channels_raw,
                output_channles,
                hidden_size,
                num_layers,
                num_classes,
                activation_fn,
                dropout,
                output_c_multip
            ).to(device)
        case ModelType.MLP:
            model = MLP(
                input_channels_extracted,
                output_channles,
                num_classes,
                activation_fn,
                dropout,
                output_c_multip
            ).to(device)

    return model

In [14]:
def train_loso(
    X_train_raw, X_test_raw,
    X_train_extracted, X_test_extracted,
    y_train, y_test, 
    groups,
    subject_idx, test_subject,
    settings, metadata,
):
    # ------------------------------------------------------------------
    # Convert to CPU tensors
    # ------------------------------------------------------------------
    if X_train_raw is not None:
        X_train_raw = torch.tensor(X_train_raw, dtype=torch.float32)
        X_test_raw = torch.tensor(X_test_raw, dtype=torch.float32)

    if X_train_extracted is not None:
        X_train_extracted = torch.tensor(X_train_extracted, dtype=torch.float32)
        X_test_extracted = torch.tensor(X_test_extracted, dtype=torch.float32)

    y_train = torch.tensor(y_train, dtype=torch.long)

    # ------------------------------------------------------------------
    # Split train / validation
    # ------------------------------------------------------------------
    split_source = X_train_raw if X_train_raw is not None else X_train_extracted

    split_np = split_source.numpy()
    y_train_np = y_train.numpy()

    gss = GroupShuffleSplit(
        n_splits=1,
        test_size=0.2,
        random_state=42 + subject_idx,
    )

    train_idx, val_idx = next(
        gss.split(split_np, y_train_np, groups)
    )

    assert len(groups) == len(split_np)

    # ------------------------------------------------------------------
    # Slice data
    # ------------------------------------------------------------------
    X_train_main_raw = None
    X_val_raw = None

    X_train_main_extracted = None
    X_val_extracted = None

    if X_train_raw is not None:
        X_train_main_raw = X_train_raw[train_idx]
        X_val_raw = X_train_raw[val_idx]

    if X_train_extracted is not None:
        X_train_main_extracted = X_train_extracted[train_idx]
        X_val_extracted = X_train_extracted[val_idx]

    y_train_main = y_train[train_idx]
    y_val = y_train[val_idx]

    if metadata["eval_metric"] == EvalMetric.custom:
        y_val = y_val.float()


    if X_train_main_raw is None:
        X_train_main_raw = torch.empty(
            (len(y_train_main), 0, 0),
            dtype=torch.float32
        )
    
    if X_val_raw is None:
        X_val_raw = torch.empty(
            (len(y_val), 0, 0),
            dtype=torch.float32
        )
    
    if X_train_main_extracted is None:
        X_train_main_extracted = torch.empty(
            (len(y_train_main), 0),
            dtype=torch.float32
        )
    
    if X_val_extracted is None:
        X_val_extracted = torch.empty(
            (len(y_val), 0),
            dtype=torch.float32
        )
    # ------------------------------------------------------------------
    # Model
    # ------------------------------------------------------------------
    model = get_model(
        settings["model settings"],
        X_train_main_raw,
        X_train_main_extracted,
        y_train_main,
    )

    optimizer = torch.optim.Adam(
        model.parameters(),
        lr=settings["training settings"]["learning rate"],
        weight_decay=settings["training settings"]["weight decay"],
    )

    loader, loss_fn = get_loader(
        settings["training settings"],
        X_train_main_raw, X_train_main_extracted, y_train_main,
    )

    # ------------------------------------------------------------------
    # Training
    # ------------------------------------------------------------------
    best_val = -1
    best_state = copy.deepcopy(model.state_dict())
    patience_counter = 0

    for epoch in range(settings["training settings"]["epoch"]):

        train_epoch(
            model,loader, optimizer, loss_fn,
            settings["training settings"]["accumulation_steps"],
        )

        val_score = validate_epoch(
            model,
            X_val_raw, X_val_extracted, y_val,
            metadata
        )

        if val_score > best_val:
            best_val = val_score
            best_state = copy.deepcopy(model.state_dict())
            patience_counter = 0
        else:
            patience_counter += 1

        if patience_counter >= settings["training settings"]["patience"]:
            break

    print(epoch - patience_counter)

    model.load_state_dict(best_state)

    # ------------------------------------------------------------------
    # Test
    # ------------------------------------------------------------------
    model.eval()

    with torch.no_grad():

        if X_test_raw is None:
            X_test_raw = torch.empty((len(y_test), 0, 0))
        
        if X_test_extracted is None:
            X_test_extracted = torch.empty((len(y_test), 0))
            
        y_test = torch.tensor(y_test, dtype=torch.float32)
        
        logits = model_forward(
            model,
            X_test_raw.to(device),
            X_test_extracted.to(device),
        )

        prob = torch.softmax(logits, dim=1).cpu()
        
        _, preds = torch.max(prob, dim=1)

        preds = preds.cpu()
        y_test = y_test.cpu()
        
        acc = compute_accuracy(y_test,preds)

    return acc

In [15]:
def normalize_train_test_raw(X_train, X_test, eps=1e-8):
    mean = X_train.mean(axis=(0, 1), keepdims=True)
    std = X_train.std(axis=(0, 1), keepdims=True)

    X_train = (X_train - mean) / (std + eps)
    X_test = (X_test - mean) / (std + eps)

    return X_train, X_test

def normalize_train_test_engineered(X_train, X_test, eps=1e-8):
    mean = X_train.mean(axis=0, keepdims=True)
    std = X_train.std(axis=0, keepdims=True)

    X_train = (X_train - mean) / (std + eps)
    X_test = (X_test - mean) / (std + eps)

    return X_train, X_test

In [16]:
def save_results(study, trial):
    study.trials_dataframe().to_csv(
        f"{study_name}_trials.csv",
        index=False
    )

In [17]:
def printParmNum(X_train, X_test, y_train, y_test, groups, subject_idx, settings):
    X_train = torch.tensor(X_train, dtype=torch.float32, device=device)
    y_train = torch.tensor(y_train, dtype=torch.long, device=device)

    X_test = torch.tensor(X_test, dtype=torch.float32, device=device)
    y_test = torch.tensor(y_test, dtype=torch.long, device=device)

    # ----------------------------
    # validation split
    # ----------------------------
    X_train_np = X_train.cpu().numpy()
    y_train_np = y_train.cpu().numpy()

    gss = GroupShuffleSplit(
        n_splits=1,
        test_size=0.2,
        random_state=42 + subject_idx
    )

    train_idx, val_idx = next(
        gss.split(X_train_np, y_train_np, groups)
    )
    
    assert len(groups) == len(X_train_np)
    
    X_train_main = X_train_np[train_idx]
    y_train_main = y_train_np[train_idx]

    X_val = X_train_np[val_idx]
    y_val = y_train_np[val_idx]

    X_train_main = torch.tensor(X_train_main, dtype=torch.float32)
    X_val = torch.tensor(X_val, dtype=torch.float32)

    y_train_main = torch.tensor(y_train_main, dtype=torch.long)
    y_val = torch.tensor(y_val, dtype=torch.long)

    model_settings = settings["model settings"]
    training_settings = settings["training settings"]
    
    lr_range = training_settings["learning rate"]
    wd_range = training_settings["weight decay"]
    
    match model_settings["activation_fn"]:
        case "relu":
            activation_fn = nn.ReLU
        case "gelu":
            activation_fn = nn.GELU

    model = CNN(
        X_train.shape[-1],
        num_classes=len(torch.unique(y_train)),
        activation_fn=activation_fn
    ).to(device)
    print("CNN: ")
    print(f"\t{count_parameters(model)} Params")

    model = LSTM(
        X_train.shape[-1],
        num_classes=len(torch.unique(y_train)),
    ).to(device)
    print("LSTM: ")
    print(f"\t{count_parameters(model)} Params")
    model = CNN_LSTM(
        X_train.shape[-1], 
        len(torch.unique(y_train)),
        activation_fn=activation_fn,
        dropout=model_settings["dropout"]
    ).to(device)
    print("CNN_LSTM: ")
    print(f"\t{count_parameters(model)} Params")

    model = LSTM_CNN(
        X_train.shape[-1],
        len(torch.unique(y_train)),
        activation_fn=activation_fn,
        dropout=model_settings["dropout"]
    ).to(device)
    print("LSTM_CNN: ")
    print(f"\t{count_parameters(model)} Params")

    model = CNN_LSTM_Fusion(
        X_train.shape[-1],
        len(torch.unique(y_train)),
        activation_fn=activation_fn,
        dropout=model_settings["dropout"]
    ).to(device)
    print("CNN_LSTM_Fusion: ")
    print(f"\t{count_parameters(model)} Params")

In [18]:
study_name = "lstm_full_data_search"

In [18]:
def objective(trial):
    metadata = {
        "eval_metric": EvalMetric.accuracy,
    }
    
    # settings = {
    #     "training settings": {
    #         "learning rate": trial.suggest_float("lr", 0.00005, 0.00012, log=True),
    #         "weight decay": trial.suggest_float("wd", 0.00009, 0.00012, log=True),
    
    #         "patience": 15,
    #         "epoch": 150,
            
    #         "weight": None,
    #         "label smoothing": trial.suggest_float("ls", 0.40, 0.5),
    #         "batch size": trial.suggest_categorical("batch", [512]),
    #         "accumulation_steps": trial.suggest_categorical("accumulation", [2, 4]),
    #     },
    #     "model settings": {
    #         "model": ModelType.CNN_LSTM,
    #         "activation_fn": "gelu",
    #         "dropout": trial.suggest_float("dp", 0.14, 0.17), 
    #         "output_channels": trial.suggest_categorical("out_ch", [128, 256]),
    #         "output_c_multip": trial.suggest_categorical("out_ch_mult", [1]),
    #         "hidden_size": trial.suggest_categorical("h_size", [64]),
    #         "num_layers": trial.suggest_categorical("n_layers", [1, 2]),
    #     }
    # }

    # settings = {
    #     "training settings": {
    #         "learning rate": trial.suggest_float("lr", 0.000005, 0.00008, log=True),
    #         "weight decay": trial.suggest_float("wd", 0.00005, 0.0002, log=True),
    
    #         "patience": 15,
    #         "epoch": 150,
            
    #         "weight": None,
    #         "label smoothing": trial.suggest_float("ls", 0.32, 0.43),
    #         "batch size": trial.suggest_categorical("batch", [256, 512, 1024]),
    #         "accumulation_steps": trial.suggest_categorical("accumulation", [2, 4]),
    #     },
    #     "model settings": {
    #         "model": ModelType.CNN_LSTM_Fusion,
    #         "activation_fn": "gelu",
    #         "dropout": trial.suggest_float("dp", 0.25, 0.34), 
    #         "output_channels": trial.suggest_categorical("out_ch", [128, 256]),
    #         "output_c_multip": trial.suggest_categorical("out_ch_mult", [1, 2]),
    #         "hidden_size": trial.suggest_categorical("h_size", [32, 64, 128, 256]),
    #         "num_layers": trial.suggest_categorical("n_layers", [1, 2]),
    #     }
    # }

    # settings = {
    #     "training settings": {
    #         "learning rate": trial.suggest_float("lr", 0.00005, 0.00009, log=True),
    #         "weight decay": trial.suggest_float("wd", 0.0002, 0.00021, log=True),
    
    #         "patience": 15,
    #         "epoch": 150,
            
    #         "weight": None,
    #         "label smoothing": trial.suggest_float("ls", 0.01, 0.39),
    #         "batch size": trial.suggest_categorical("batch", [64, 128, 256]),
    #         "accumulation_steps": trial.suggest_categorical("accumulation", [1, 2, 4]),
    #     },
    #     "model settings": {
    #         "model": ModelType.CNN,
    #         "activation_fn": "gelu",
    #         "dropout": trial.suggest_float("dp", 0.16, 0.48), 
    #         "output_channels": trial.suggest_categorical("out_ch", [32, 64, 128, 256]),
    #         "output_c_multip": trial.suggest_categorical("out_ch_mult", [1, 2]),
    #         "hidden_size": 0,
    #         "num_layers": 0,
    #     }
    # }
    
    # settings = {
    #     "training settings": {
    #         "learning rate": trial.suggest_float("lr", 0.00005, 0.006, log=True),
    #         "weight decay": trial.suggest_float("wd", 0.00205, 0.0050, log=True),
    
    #         "patience": 15,
    #         "epoch": 150,
            
    #         "weight": None,
    #         "label smoothing": trial.suggest_float("ls", 0.2882, 0.8022),
    #         "batch size": trial.suggest_categorical("batch", [64, 128, 256, 512]),
    #         "accumulation_steps": 10,
    #     },
    #     "model settings": {
    #         "model": ModelType.CNN_Transformer,
    #         "activation_fn": "gelu",
    #         "dropout": trial.suggest_float("dp", 0.62, 1), 
    #         "output_channels": trial.suggest_categorical("out_ch", [32, 64, 128, 256, 512]),
    #         "hidden_size": trial.suggest_categorical("h_size", [32, 64, 128, 256, 512, 1024]),
    #         "num_layers": trial.suggest_categorical("n_layers", [1, 2, 3]),
    #     }
    # }

    settings = {
        "training settings": {
            "learning rate": trial.suggest_float("lr", 0.00005, 0.00008423283251774466, log=True),
            "weight decay": trial.suggest_float("wd", 0.0000774864039638157, 0.00010670412710619434, log=True),
    
            "patience": 15,
            "epoch": 150,
            
            "weight": None,
            "label smoothing": trial.suggest_float("ls", 0.10415540096716684, 0.3529085314624379),
            "batch size": trial.suggest_categorical("batch", [64, 128, 256]),
            "accumulation_steps": trial.suggest_categorical("accumulation", [1, 2, 4]),
        },
        "model settings": {
            "model": ModelType.LSTM,
            "activation_fn": "gelu",
            "dropout": trial.suggest_float("dp", 0.0522868546134969, 0.1306356205341424), 
            "output_channels": trial.suggest_categorical("out_ch", [32, 64, 128, 256]),
            "output_c_multip": trial.suggest_categorical("out_ch_mult", [2]),
            "hidden_size": trial.suggest_categorical("h_size", [32, 64, 128]),
            "num_layers": trial.suggest_categorical("n_layers", [1, 2, 3]),
        }
    }

    model_type = settings["model settings"]["model"]
    use_raw = model_type in [
        ModelType.CNN,
        ModelType.LSTM,
        ModelType.CNN_LSTM,
        ModelType.LSTM_CNN,
        ModelType.CNN_Transformer,
        ModelType.CNN_LSTM_Fusion,
    ]
    
    use_engineered = model_type in [
        ModelType.MLP,
        ModelType.CNN_Transformer,
    ]
    files_suffix = "_raw" if use_raw else "_extracted"
    
    cache_hash = "7f614e721ed6b83d1d2a0c91f5a7c0ef"
    cache_dir = f"{data_path}Final Training Data/Windowed Data/{cache_hash}"
    
    subjects = get_subjects(cache_dir, use_raw, use_engineered)

    accs = []
    for subject_idx, test_subject in enumerate(subjects):
        start = time.perf_counter()
        train_subjects = [s for s in subjects if s != test_subject]
        test_subject = test_subject.removesuffix(files_suffix)
        print(f"\nSubject {subject_idx}: {test_subject}")
        print("Loading data...")
    
        X_train_raw_list = []
        X_train_engineered_list = []
        y_train_list = []
        groups = []
        
        for s in train_subjects:
            s = s.removesuffix(files_suffix)
            X_s_raw, X_s_engineered, y_s = load_subject(cache_dir, s, use_raw, use_engineered)
            if use_raw:
                X_train_raw_list.append(X_s_raw)
            
            if use_engineered:
                X_train_engineered_list.append(X_s_engineered)
            
            y_train_list.append(y_s)
            groups.extend([s] * len(X_s_raw if X_s_raw is not None else X_s_engineered))
            
        X_train_raw = np.concatenate(X_train_raw_list) if use_raw else None
        X_train_engineered = np.concatenate(X_train_engineered_list) if use_engineered else None
        y_train = np.concatenate(y_train_list)
        groups = np.array(groups)
        
        X_test_raw, X_test_engineered, y_test = load_subject(cache_dir, test_subject, use_raw, use_engineered)
        if use_raw:
            X_train_raw, X_test_raw = normalize_train_test_raw(
                X_train_raw,
                X_test_raw
            )
        
        if use_engineered:
            X_train_engineered, X_test_engineered = normalize_train_test_engineered(
                X_train_engineered,
                X_test_engineered
            )
        
        print("Training...")
        acc = train_loso(
                X_train_raw, X_test_raw,
                X_train_engineered, X_test_engineered,
                y_train, y_test,
                groups,
                subject_idx, test_subject,
                settings, metadata
            )
        accs.append(acc)

        elapsed = time.perf_counter() - start

        print(f"Time: {elapsed:.2f} seconds")
    return sum(accs) / len(accs)



study = optuna.create_study(
    study_name=study_name,
    storage="sqlite:///optuna.db",
    load_if_exists=True,
    direction="maximize",
)
study.optimize(objective, n_trials=100, callbacks=[save_results])

[I 2026-08-24 21:02:07,809] Using an existing study with name 'lstm_full_data_search' instead of creating a new one.



Subject 0: 0fc57dc8-de68-4b41-bf69-59a5a5f2d27e
Loading data...
Training...
1
Time: 45.07 seconds

Subject 1: 10b8d8be-39de-43e7-9260-ba4cc724d4ae
Loading data...
Training...
0
Time: 38.91 seconds

Subject 2: 1279952d-14d4-4e77-9010-a000dd546bcd
Loading data...
Training...
1
Time: 38.16 seconds

Subject 3: 1477b633-8ef3-4123-9ecc-f2ee90d13d8c
Loading data...
Training...
12
Time: 61.66 seconds

Subject 4: 383b0fe8-654a-4f4f-84cd-375470069789
Loading data...
Training...
0
Time: 36.62 seconds

Subject 5: 3b1c507b-c290-4250-95c9-e21d6c52e3f2
Loading data...
Training...
4
Time: 46.68 seconds

Subject 6: 5bcd3401-e378-482f-b9c6-37edbad97d1a
Loading data...
Training...
6
Time: 58.17 seconds

Subject 7: 63d66e1c-434f-4f23-8e74-abd8dedc0e43
Loading data...
Training...
2
Time: 42.72 seconds

Subject 8: 63efffc7-e347-47fb-9615-f7da183d8792
Loading data...
Training...
0
Time: 36.26 seconds

Subject 9: 6f32fcb3-9e76-448d-9de8-799e376fdea6
Loading data...
Training...
3
Time: 44.56 seconds

Subject 

[I 2026-08-24 21:19:58,715] Trial 34 finished with value: 0.5082285814415488 and parameters: {'lr': 6.819397747214966e-05, 'wd': 9.626048418639767e-05, 'ls': 0.24374217468548104, 'batch': 64, 'accumulation': 1, 'dp': 0.07543723673349759, 'out_ch': 256, 'out_ch_mult': 2, 'h_size': 128, 'n_layers': 2}. Best is trial 20 with value: 0.5137431429932652.


7
Time: 53.10 seconds

Subject 0: 0fc57dc8-de68-4b41-bf69-59a5a5f2d27e
Loading data...
Training...
1
Time: 41.15 seconds

Subject 1: 10b8d8be-39de-43e7-9260-ba4cc724d4ae
Loading data...
Training...
2
Time: 41.76 seconds

Subject 2: 1279952d-14d4-4e77-9010-a000dd546bcd
Loading data...
Training...
0
Time: 36.44 seconds

Subject 3: 1477b633-8ef3-4123-9ecc-f2ee90d13d8c
Loading data...
Training...
3
Time: 44.65 seconds

Subject 4: 383b0fe8-654a-4f4f-84cd-375470069789
Loading data...
Training...
4
Time: 47.40 seconds

Subject 5: 3b1c507b-c290-4250-95c9-e21d6c52e3f2
Loading data...
Training...
10
Time: 60.19 seconds

Subject 6: 5bcd3401-e378-482f-b9c6-37edbad97d1a
Loading data...
Training...
0
Time: 42.25 seconds

Subject 7: 63d66e1c-434f-4f23-8e74-abd8dedc0e43
Loading data...
Training...
0
Time: 38.05 seconds

Subject 8: 63efffc7-e347-47fb-9615-f7da183d8792
Loading data...
Training...
4
Time: 45.27 seconds

Subject 9: 6f32fcb3-9e76-448d-9de8-799e376fdea6
Loading data...
Training...
0
Time: 3

[I 2026-08-24 21:37:50,092] Trial 35 finished with value: 0.5087297937503847 and parameters: {'lr': 6.90835657399646e-05, 'wd': 9.796064102501119e-05, 'ls': 0.2803812713813584, 'batch': 64, 'accumulation': 1, 'dp': 0.09478320927449319, 'out_ch': 256, 'out_ch_mult': 2, 'h_size': 128, 'n_layers': 2}. Best is trial 20 with value: 0.5137431429932652.


...

2
Time: 32.03 seconds

Subject 0: 0fc57dc8-de68-4b41-bf69-59a5a5f2d27e
Loading data...
Training...
1
Time: 40.15 seconds

Subject 1: 10b8d8be-39de-43e7-9260-ba4cc724d4ae
Loading data...
Training...
0
Time: 37.38 seconds

Subject 2: 1279952d-14d4-4e77-9010-a000dd546bcd
Loading data...
Training...
0
Time: 35.97 seconds

Subject 3: 1477b633-8ef3-4123-9ecc-f2ee90d13d8c
Loading data...
Training...
6
Time: 49.33 seconds

Subject 4: 383b0fe8-654a-4f4f-84cd-375470069789
Loading data...
Training...
2
Time: 42.23 seconds

Subject 5: 3b1c507b-c290-4250-95c9-e21d6c52e3f2
Loading data...
Training...
4
Time: 46.26 seconds

Subject 6: 5bcd3401-e378-482f-b9c6-37edbad97d1a
Loading data...
Training...
22
Time: 96.53 seconds

Subject 7: 63d66e1c-434f-4f23-8e74-abd8dedc0e43
Loading data...
Training...
1
Time: 41.02 seconds

Subject 8: 63efffc7-e347-47fb-9615-f7da183d8792
Loading data...
Training...
9
Time: 56.38 seconds

Subject 9: 6f32fcb3-9e76-448d-9de8-799e376fdea6
Loading data...
Training...
1
Time: 4

[I 2026-08-25 03:42:46,281] Trial 61 finished with value: 0.4967468603541967 and parameters: {'lr': 6.51312282146205e-05, 'wd': 7.950141220982713e-05, 'ls': 0.17003762205312448, 'batch': 64, 'accumulation': 1, 'dp': 0.0833363032128091, 'out_ch': 128, 'out_ch_mult': 2, 'h_size': 128, 'n_layers': 2}. Best is trial 39 with value: 0.515244207268362.


0
Time: 37.47 seconds

Subject 0: 0fc57dc8-de68-4b41-bf69-59a5a5f2d27e
Loading data...
Training...
6
Time: 48.50 seconds

Subject 1: 10b8d8be-39de-43e7-9260-ba4cc724d4ae
Loading data...
Training...
0
Time: 34.82 seconds

Subject 2: 1279952d-14d4-4e77-9010-a000dd546bcd
Loading data...
Training...
4
Time: 40.77 seconds

Subject 3: 1477b633-8ef3-4123-9ecc-f2ee90d13d8c
Loading data...
Training...
2
Time: 37.10 seconds

Subject 4: 383b0fe8-654a-4f4f-84cd-375470069789
Loading data...
Training...
2
Time: 38.66 seconds

Subject 5: 3b1c507b-c290-4250-95c9-e21d6c52e3f2
Loading data...
Training...
0
Time: 35.91 seconds

Subject 6: 5bcd3401-e378-482f-b9c6-37edbad97d1a
Loading data...
Training...
0
Time: 32.93 seconds

Subject 7: 63d66e1c-434f-4f23-8e74-abd8dedc0e43
Loading data...
Training...
9
Time: 54.46 seconds

Subject 8: 63efffc7-e347-47fb-9615-f7da183d8792
Loading data...
Training...
2
Time: 38.47 seconds

Subject 9: 6f32fcb3-9e76-448d-9de8-799e376fdea6
Loading data...
Training...
1
Time: 38

[W 2026-08-25 03:49:37,664] Trial 62 failed with parameters: {'lr': 6.594186957093237e-05, 'wd': 8.189109884605027e-05, 'ls': 0.26994251161612043, 'batch': 64, 'accumulation': 2, 'dp': 0.05615026593294159, 'out_ch': 32, 'out_ch_mult': 2, 'h_size': 32, 'n_layers': 2} because of the following error: KeyboardInterrupt().
Traceback (most recent call last):
  File "/usr/local/lib/python3.10/dist-packages/optuna/study/_optimize.py", line 206, in _run_trial
    value_or_values = func(trial)
  File "/tmp/ipykernel_1950/3728832797.py", line 190, in objective
    acc = train_loso(
  File "/tmp/ipykernel_1950/3544507715.py", line 119, in train_loso
    train_epoch(
  File "/tmp/ipykernel_1950/2161065454.py", line 17, in train_epoch
    loss.backward()
  File "/usr/local/lib/python3.10/dist-packages/torch/_tensor.py", line 630, in backward
    torch.autograd.backward(
  File "/usr/local/lib/python3.10/dist-packages/torch/autograd/__init__.py", line 364, in backward
    _engine_run_backward(
  File

KeyboardInterrupt: 

In [19]:
CSV_PATH = f"{study_name}_trials.csv"
SCORE_COLUMN = "value"
 
df = pl.read_csv(CSV_PATH)

# Keep only completed trials (if the column exists)
if "state" in df.columns:
    df = df.filter(pl.col("state") == "COMPLETE")

best_values = df[ df["value"].arg_max() ]
best_score = best_values.select(pl.col("value")).item()
tolerance = 0.000

while True:
    best_df = df.filter(pl.col("value") >= best_score - tolerance)

    if best_df.height >= 10:
        break

    tolerance += 0.001

best_df = df.filter(
    pl.col(SCORE_COLUMN) >= best_score - tolerance
)

param_columns = [c for c in best_values.columns if c.startswith("params_")]

print(f"Best score: {best_score:.6f}")
for param in param_columns:
    val = (
        best_values.select(
            pl.col(param),
        )
        .row(0)
    )

    print(f"{param.removeprefix('params_'):20} "
          f"{val}")

print(f"Trials kept: {best_df.height} / {df.height}")
print(f"Min score: {best_score - tolerance}")
param_columns = [c for c in df.columns if c.startswith("params_")]

print("\nParameter ranges:")
for param in param_columns:
    min_val, max_val = (
        best_df.select(
            pl.col(param).min().alias("min"),
            pl.col(param).max().alias("max"),
        )
        .row(0)
    )

    print(f"{param.removeprefix('params_'):20} "
          f"min={min_val}    max={max_val}")

FileNotFoundError: No such file or directory (os error 2): lstm_full_data_search_trials.csv